# Machine Learning Weather Models

Train và so sánh Random Forest, XGBoost cho bài toán dự báo `temperature_2m`.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ttest_rel
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, cross_validate
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data/processed/Hanoi_final.csv").exists() and (PROJECT_ROOT.parent / "data/processed/Hanoi_final.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data/processed/Hanoi_final.csv"
OUTPUT_DIR = PROJECT_ROOT / "train_model/artifacts/ml"

TARGET_COLUMN = "temperature_2m"
HORIZON_HOURS = 1
TEST_SIZE = 0.2
RANDOM_STATE = 42
CV_SPLITS = 3
TUNING_ITERATIONS = 6

LAGS = [1, 2, 3, 6, 12, 24, 48, 72, 168]
ROLLING_WINDOWS = [3, 6, 12, 24]

print(DATA_PATH)

## 1. Đọc final dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df = df.dropna(subset=["time"]).sort_values("time").reset_index(drop=True)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print(f"Time: {df['time'].min()} -> {df['time'].max()}")
df.head()

## 2. Feature Engineering cho ML

In [ ]:
def add_forecast_time_features(dataframe: pd.DataFrame) -> pd.DataFrame:
    result = dataframe.copy()
    forecast_time = result["time"] + pd.to_timedelta(HORIZON_HOURS, unit="h")
    result["forecast_hour"] = forecast_time.dt.hour
    result["forecast_day_of_week"] = forecast_time.dt.dayofweek
    result["forecast_day_of_year"] = forecast_time.dt.dayofyear
    result["forecast_month"] = forecast_time.dt.month
    result["forecast_hour_sin"] = np.sin(2 * np.pi * result["forecast_hour"] / 24)
    result["forecast_hour_cos"] = np.cos(2 * np.pi * result["forecast_hour"] / 24)
    result["forecast_month_sin"] = np.sin(2 * np.pi * result["forecast_month"] / 12)
    result["forecast_month_cos"] = np.cos(2 * np.pi * result["forecast_month"] / 12)
    result["forecast_day_of_year_sin"] = np.sin(2 * np.pi * result["forecast_day_of_year"] / 366)
    result["forecast_day_of_year_cos"] = np.cos(2 * np.pi * result["forecast_day_of_year"] / 366)
    return result


def create_ml_dataset(dataframe: pd.DataFrame) -> tuple[pd.DataFrame, list[str], str]:
    data = add_forecast_time_features(dataframe)
    data[TARGET_COLUMN] = pd.to_numeric(data[TARGET_COLUMN], errors="coerce")

    shifted_target = data[TARGET_COLUMN].shift(1)
    for lag in LAGS:
        data[f"{TARGET_COLUMN}_lag_{lag}h"] = data[TARGET_COLUMN].shift(lag)

    for window in ROLLING_WINDOWS:
        data[f"{TARGET_COLUMN}_rolling_mean_{window}h"] = shifted_target.rolling(window, min_periods=window).mean()
        data[f"{TARGET_COLUMN}_rolling_std_{window}h"] = shifted_target.rolling(window, min_periods=window).std()

    label_column = f"{TARGET_COLUMN}_future_{HORIZON_HOURS}h"
    data[label_column] = data[TARGET_COLUMN].shift(-HORIZON_HOURS)

    numeric_columns = data.select_dtypes(include=[np.number]).columns.tolist()
    feature_columns = [column for column in numeric_columns if column != label_column]
    dataset = data[["time", *feature_columns, label_column]].dropna().reset_index(drop=True)

    constant_features = [column for column in feature_columns if dataset[column].nunique(dropna=False) <= 1]
    feature_columns = [column for column in feature_columns if column not in constant_features]
    dataset = dataset[["time", *feature_columns, label_column]]
    return dataset, feature_columns, label_column


dataset, feature_columns, label_column = create_ml_dataset(df)
print(f"Dataset rows: {len(dataset):,}")
print(f"Feature count: {len(feature_columns):,}")
dataset.head()

## 3. Train/Test split theo thời gian

In [ ]:
split_index = int(len(dataset) * (1 - TEST_SIZE))
train_df = dataset.iloc[:split_index].copy()
test_df = dataset.iloc[split_index:].copy()

X_train = train_df[feature_columns]
y_train = train_df[label_column]
X_test = test_df[feature_columns]
y_test = test_df[label_column]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Train time: {train_df['time'].min()} -> {train_df['time'].max()}")
print(f"Test time: {test_df['time'].min()} -> {test_df['time'].max()}")

## 4. Hàm đánh giá

In [ ]:
def evaluate_model(model, X_eval: pd.DataFrame, y_eval: pd.Series) -> dict[str, float]:
    pred = model.predict(X_eval)
    denominator = np.maximum(np.abs(y_eval.to_numpy()), 1e-8)
    return {
        "MAE": mean_absolute_error(y_eval, pred),
        "Median AE": median_absolute_error(y_eval, pred),
        "RMSE": np.sqrt(mean_squared_error(y_eval, pred)),
        "R2": r2_score(y_eval, pred),
        "MAPE (%)": np.mean(np.abs((y_eval.to_numpy() - pred) / denominator)) * 100,
    }


def format_metrics(metrics: pd.DataFrame) -> pd.DataFrame:
    formatted = metrics.copy()
    for column in ["MAE", "Median AE", "RMSE", "R2", "MAPE (%)"]:
        formatted[column] = formatted[column].round(4)
    return formatted

## 5. Baseline + TimeSeriesSplit CV

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "XGBoost": XGBRegressor(
        objective="reg:squarederror",
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
        eval_metric="rmse",
    ),
}

cv = TimeSeriesSplit(n_splits=CV_SPLITS)
scoring = {"mae": "neg_mean_absolute_error", "mse": "neg_mean_squared_error", "r2": "r2"}
cv_fold_mae = {}
cv_rows = []

for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    mae_scores = -scores["test_mae"]
    rmse_scores = np.sqrt(-scores["test_mse"])
    r2_scores = scores["test_r2"]
    cv_fold_mae[name] = mae_scores
    cv_rows.append({
        "Model": name,
        "CV MAE mean": mae_scores.mean(),
        "CV MAE std": mae_scores.std(),
        "CV RMSE mean": rmse_scores.mean(),
        "CV R2 mean": r2_scores.mean(),
    })

cv_summary = pd.DataFrame(cv_rows).sort_values("CV MAE mean")
cv_summary.round(4)

## 6. Hyperparameter Tuning

In [ ]:
rf_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions={
        "n_estimators": [150, 250, 350],
        "max_depth": [10, 15, 20, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", 0.7, 1.0],
    },
    n_iter=TUNING_ITERATIONS,
    cv=cv,
    scoring="neg_mean_absolute_error",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=1,
)

xgb_search = RandomizedSearchCV(
    estimator=XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist",
    ),
    param_distributions={
        "n_estimators": [250, 400, 550],
        "max_depth": [3, 4, 5, 6],
        "learning_rate": [0.03, 0.05, 0.08],
        "subsample": [0.75, 0.9, 1.0],
        "colsample_bytree": [0.75, 0.9, 1.0],
        "min_child_weight": [1, 3, 5],
        "reg_lambda": [1, 5, 10],
    },
    n_iter=TUNING_ITERATIONS,
    cv=cv,
    scoring="neg_mean_absolute_error",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=1,
)

rf_search.fit(X_train, y_train)
xgb_search.fit(X_train, y_train)

print("RF best params:", rf_search.best_params_)
print("XGB best params:", xgb_search.best_params_)

## 7. Đánh giá cuối trên test set

In [ ]:
tuned_models = {
    "Random Forest Tuned": rf_search.best_estimator_,
    "XGBoost Tuned": xgb_search.best_estimator_,
}
all_models = {**models, **tuned_models}

test_rows = []
for name, model in all_models.items():
    model.fit(X_train, y_train)
    metrics = evaluate_model(model, X_test, y_test)
    metrics["Model"] = name
    test_rows.append(metrics)

test_results = pd.DataFrame(test_rows).set_index("Model").sort_values("RMSE")
format_metrics(test_results)

## 8. Kiểm định thống kê CV

In [ ]:
for name, model in tuned_models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    cv_fold_mae[name] = -scores["test_mae"]

stat_tests = []
for baseline_name, tuned_name in [("Random Forest", "Random Forest Tuned"), ("XGBoost", "XGBoost Tuned")]:
    statistic, p_value = ttest_rel(cv_fold_mae[baseline_name], cv_fold_mae[tuned_name])
    stat_tests.append({
        "Comparison": f"{baseline_name} vs {tuned_name}",
        "Baseline CV MAE": cv_fold_mae[baseline_name].mean(),
        "Tuned CV MAE": cv_fold_mae[tuned_name].mean(),
        "Mean MAE improvement": cv_fold_mae[baseline_name].mean() - cv_fold_mae[tuned_name].mean(),
        "paired t-test p-value": p_value,
    })

pd.DataFrame(stat_tests).round(4)

## 9. Feature Importance và Error Analysis

In [ ]:
best_model_name = test_results.index[0]
best_model = all_models[best_model_name]
best_predictions = best_model.predict(X_test)

importance_df = pd.DataFrame({
    "feature": feature_columns,
    "importance": best_model.feature_importances_,
}).sort_values("importance", ascending=False)

importance_df.head(20)

In [ ]:
predictions_df = test_df[["time"]].copy()
predictions_df["actual"] = y_test.to_numpy()
predictions_df["predicted"] = best_predictions
predictions_df["error"] = predictions_df["actual"] - predictions_df["predicted"]
predictions_df["absolute_error"] = predictions_df["error"].abs()
predictions_df["hour"] = predictions_df["time"].dt.hour
predictions_df["month"] = predictions_df["time"].dt.month

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=importance_df.head(15), x="importance", y="feature", ax=axes[0])
axes[0].set_title("Top feature importance")
sns.lineplot(data=predictions_df.groupby("hour", as_index=False)["absolute_error"].mean(), x="hour", y="absolute_error", marker="o", ax=axes[1])
axes[1].set_title("MAE by hour")
sns.scatterplot(data=predictions_df.sample(min(5000, len(predictions_df)), random_state=RANDOM_STATE), x="actual", y="predicted", alpha=0.35, ax=axes[2])
axes[2].set_title("Actual vs Predicted")
plt.tight_layout()

## 10. Lưu kết quả

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

model_file = OUTPUT_DIR / f"best_ml_{best_model_name.lower().replace(' ', '_')}_{TARGET_COLUMN}_{timestamp}.joblib"
metrics_file = OUTPUT_DIR / f"ml_metrics_{TARGET_COLUMN}_{timestamp}.json"
predictions_file = OUTPUT_DIR / f"ml_predictions_{TARGET_COLUMN}_{timestamp}.csv"
importance_file = OUTPUT_DIR / f"ml_feature_importance_{TARGET_COLUMN}_{timestamp}.csv"

joblib.dump({
    "model": best_model,
    "model_name": best_model_name,
    "feature_columns": feature_columns,
    "label_column": label_column,
    "target": TARGET_COLUMN,
    "horizon_hours": HORIZON_HOURS,
    "lags": LAGS,
    "rolling_windows": ROLLING_WINDOWS,
}, model_file)

metrics_payload = {
    "target": TARGET_COLUMN,
    "horizon_hours": HORIZON_HOURS,
    "best_model": best_model_name,
    "test_results": test_results.to_dict(orient="index"),
}
metrics_file.write_text(json.dumps(metrics_payload, indent=2, ensure_ascii=False), encoding="utf-8")
predictions_df.to_csv(predictions_file, index=False)
importance_df.to_csv(importance_file, index=False)

print(model_file)
print(metrics_file)
print(predictions_file)
print(importance_file)